# 01 - Bronze - Ingestão S3

**Tech Challenge - Fase 3 - Big Data to Analytics**

Objetivo: conectar no S3 e subir as 3 bases brutas para a camada `bronze/` do data lake.


## 1. Importações e conexão com o S3

In [1]:
import boto3
import os

In [2]:
# Conecta ao S3 usando as credenciais do arquivo ~/.aws/credentials
s3 = boto3.client("s3", region_name="us-east-1")

# teste rapido: se as credenciais estiverem erradas/expiradas, isso ja da erro aqui
s3.list_buckets()
print("Conectado na AWS.")

Conectado na AWS.


## 2. Configuração do bucket e dos arquivos

In [17]:
# Nome do bucket S3
NOME_BUCKET = "tech-challenge-fase-3-grupo-94" 

# Pasta local onde estão os arquivos CSV originais
PASTA_ORIGEM = "." 

# mapeamento: arquivo local -> destino no S3
ARQUIVOS_PARA_SUBIR = {
    "State_of_data_BR_2023_Kaggle - df_survey_2023.csv": "bronze/state_of_data/ano=2023/",
    "Final Dataset - State of Data 2024 - Kaggle - df_survey_2024.csv": "bronze/state_of_data/ano=2024/",
    "Final Dataset - State of Data 2025-2026 - Kaggle.csv": "bronze/state_of_data/ano=2025/",
}

## 3. Criar o bucket

In [18]:
try:
    s3.head_bucket(Bucket=NOME_BUCKET)
    print(f"Bucket '{NOME_BUCKET}' ja existe.")
except s3.exceptions.ClientError:
    s3.create_bucket(Bucket=NOME_BUCKET)
    print(f"Bucket '{NOME_BUCKET}' criado.")

Bucket 'tech-challenge-fase-3-grupo-94' ja existe.


## 4. Subir bases brutas para bronze/state_of_data/ (particionado por ano)

In [19]:
for nome_arquivo, destino_s3 in ARQUIVOS_PARA_SUBIR.items():
    caminho_local = os.path.join(PASTA_ORIGEM, nome_arquivo)

    print(f"Enviando {nome_arquivo} para s3://{NOME_BUCKET}/{destino_s3}{nome_arquivo}")
    s3.upload_file(caminho_local, NOME_BUCKET, destino_s3+nome_arquivo)
    print("Upload concluído.")

Enviando State_of_data_BR_2023_Kaggle - df_survey_2023.csv para s3://tech-challenge-fase-3-grupo-94/bronze/state_of_data/ano=2023/State_of_data_BR_2023_Kaggle - df_survey_2023.csv
Upload concluído.
Enviando Final Dataset - State of Data 2024 - Kaggle - df_survey_2024.csv para s3://tech-challenge-fase-3-grupo-94/bronze/state_of_data/ano=2024/Final Dataset - State of Data 2024 - Kaggle - df_survey_2024.csv
Upload concluído.
Enviando Final Dataset - State of Data 2025-2026 - Kaggle.csv para s3://tech-challenge-fase-3-grupo-94/bronze/state_of_data/ano=2025/Final Dataset - State of Data 2025-2026 - Kaggle.csv
Upload concluído.


## 5. Criar as demais pastas base (silver/ e gold/ vazias)

In [20]:
for pasta in ["silver/", "gold/"]:
    s3.put_object(Bucket=NOME_BUCKET, Key=pasta)
    print(f"Pasta criada: s3://{NOME_BUCKET}/{pasta}")

Pasta criada: s3://tech-challenge-fase-3-grupo-94/silver/
Pasta criada: s3://tech-challenge-fase-3-grupo-94/gold/


## 6. Revisando a criação da camada bronze

In [21]:
resposta = s3.list_objects_v2(Bucket=NOME_BUCKET, Prefix="bronze/")

print("Conteudo na camada bronze/:")
for objeto in resposta.get("Contents", []):
    print(f" - {objeto['Key']}")

print("\nIngestao finalizada. A camada bronze/ nunca deve ser sobrescrita")
print("nos proximos scripts, pois ela é a fotografia do dado bruto.")

Conteudo na camada bronze/:
 - bronze/state_of_data/ano=2023/State_of_data_BR_2023_Kaggle - df_survey_2023.csv
 - bronze/state_of_data/ano=2024/Final Dataset - State of Data 2024 - Kaggle - df_survey_2024.csv
 - bronze/state_of_data/ano=2025/Final Dataset - State of Data 2025-2026 - Kaggle.csv

Ingestao finalizada. A camada bronze/ nunca deve ser sobrescrita
nos proximos scripts, pois ela é a fotografia do dado bruto.
